
# NLP projekt - nastawienie autorów artykułów

In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import GridSearchCV
import scipy.stats as stats
from sklearn.model_selection import RandomizedSearchCV
from matplotlib import pyplot as plt
from Helpers.data_set_cleaner import import_amazon_reviews_csv_file, amazon_reviews_split
from Helpers.text_processor import TextProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Początkowy trening

## Załadowanie danych do treningu

In [2]:
path = r".\..\Data\train.csv"

df = import_amazon_reviews_csv_file(path, 5000)

df

,rating,full_text
0,Neutral,more like funchuck Gave this to my dad for a g...
1,Positive,Inspiring I hope a lot of people hear this cd....
2,Positive,The best soundtrack ever to anything. I'm read...
3,Positive,Chrono Cross OST The music of Yasunori Misuda ...
4,Positive,Too good to be true Probably the greatest soun...
...,...,...
4995,Negative,Waste oF $MONEY$ Waste Of Time & Money... I fo...
4996,Positive,trying to win better This book cuts down the o...
4997,Negative,"don""t waste your money just buy the the lotter..."
4998,Negative,The odds are against you not for you. Winning ...


## Trening

In [3]:
print("[POCZATEK TWORZENIA MODELU]")
data_path = r".\..\Data\train.csv"

data_quantity = 2_000

df = import_amazon_reviews_csv_file(data_path, data_quantity)

X_raw, y = amazon_reviews_split(df)
tp = TextProcessor()
print("[PREPROCESSING (czasochlonne)]")
X_processed = tp.preprocess_series(X_raw)
print("[SPLIT DATA]")
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

# wynik z search grida dla LogisticRegression:
# {'clf__C': 1.0, 'tfidf__max_df': 0.8, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 3)}
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 3), max_df=0.8, min_df=3)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, C = 1.0))
])
print("[Trenowanie model]")
model_pipeline.fit(X_train, y_train)

print("[KONIEC TWORZENIA MODELU]")

[POCZATEK TWORZENIA MODELU]
[PREPROCESSING (czasochlonne)]
[SPLIT DATA]
[Trenowanie model]
[KONIEC TWORZENIA MODELU]


## Ocena uzyskanego modelu

In [ ]:
predictions = model_pipeline.predict(X_test)
print(classification_report(y_test, predictions))

#chyba powinno się jakoś sprawdzać czy negatywnych nie bierze jako pozytywnych, no bo jednak jak jest coś negatywne/pozytywne a klasyfikuje się jako neutralne (lub na odwrót) to nie ma tragedii
etykiety = ['Negative', 'Neutral', 'Positive']
cm = confusion_matrix(y_test, predictions, labels=etykiety)

# 3. Rysujemy wykres
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=etykiety)
disp.plot(cmap='Blues', ax=ax, values_format='d')

plt.title("Macierz Pomyłek - Analiza Błędów")
plt.show()

# Testy różnych klasyfikatorów

## LinearSVC

In [7]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(class_weight='balanced',random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3), (1, 4)],
    'tfidf__max_df': [0.85, 0.9, 0.95],
    'tfidf__min_df': [2, 3, 5],
    'clf__C': [0.1, 1.0, 10]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__C': 1.0, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2)}
0.5785286261948511
              precision    recall  f1-score   support

    Negative       0.70      0.80      0.75       152
     Neutral       0.48      0.34      0.40        82
    Positive       0.78      0.78      0.78       166

    accuracy                           0.70       400
   macro avg       0.65      0.64      0.64       400
weighted avg       0.69      0.70      0.69       400



## LogisticRegression

In [8]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3), (1,4)],
    'tfidf__max_df': [0.85, 0.8],
    'tfidf__min_df': [3, 5],
    'clf__C': [0.1, 1.0]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='precision_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__C': 1.0, 'tfidf__max_df': 0.85, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 3)}
0.5938053886098834
              precision    recall  f1-score   support

    Negative       0.72      0.78      0.75       152
     Neutral       0.46      0.39      0.42        82
    Positive       0.78      0.77      0.78       166

    accuracy                           0.70       400
   macro avg       0.65      0.65      0.65       400
weighted avg       0.69      0.70      0.69       400



## RandomForestClassifier

In [6]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'tfidf__max_df': [0.85, 0.95],
    'tfidf__min_df': [3, 5],
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [None, 30]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__max_depth': 30, 'clf__n_estimators': 50, 'tfidf__max_df': 0.85, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 3)}
0.5146443481833397
              precision    recall  f1-score   support

    Negative       0.61      0.74      0.67       152
     Neutral       0.41      0.13      0.20        82
    Positive       0.66      0.75      0.70       166

    accuracy                           0.62       400
   macro avg       0.56      0.54      0.53       400
weighted avg       0.59      0.62      0.59       400



# Trening na większych danych

# Testy projektu

In [2]:
data_path = r".\..\Data\test.csv"
data_quantity = 3_000

df = import_amazon_reviews_csv_file(data_path, data_quantity)

X_raw, y = amazon_reviews_split(df)
tp = TextProcessor()
X_processed = tp.preprocess_series(X_raw)

In [3]:
model = joblib.load(r"../Models/model3.joblib")

predictions = model.predict(X_processed)
print(y[:5])
print(predictions[:5])
print("[PREDICTIONS OF AMAZON]")
print(classification_report(y, predictions))

0    Negative
1    Positive
2    Negative
3    Negative
4    Negative
Name: rating, dtype: object
['Neutral' 'Positive' 'Negative' 'Negative' 'Negative']
[PREDICTIONS OF AMAZON]
              precision    recall  f1-score   support

    Negative       0.77      0.75      0.76      1214
     Neutral       0.41      0.44      0.43       576
    Positive       0.76      0.76      0.76      1210

    accuracy                           0.69      3000
   macro avg       0.65      0.65      0.65      3000
weighted avg       0.70      0.69      0.70      3000



In [4]:
#texts wykreowane przez gemini
texts = {
    "shouldbe_negative_1.txt": "Authorities have confirmed the death of a 34-year-old man following a fatal shooting late Friday night in the city center. Detectives are currently treating the incident as a targeted attack and are urging any witnesses to come forward. The local community remains on high alert as the suspect is still at large and considered armed and dangerous.",
    "shouldbe_negative_2.txt": "A devastating collision involving two semi-trucks and several passenger vehicles resulted in at least four fatalities and dozens of severe injuries yesterday morning. Emergency responders worked for hours to extract victims from the mangled wreckage. Slippery road conditions and heavy fog are believed to be the primary contributing factors to this horrific tragedy.",
    "shouldbe_negative_3.txt": "Sudden flash flooding swept through the lower valley early Tuesday, destroying dozens of homes and claiming at least ten lives in its wake. Rescue teams are currently recovering bodies from the mud and debris while desperately searching for several individuals still reported missing. The town is now facing an unprecedented humanitarian crisis.",
    "shouldbe_negative_4.txt": "A massive financial fraud scheme has collapsed, leaving hundreds of elderly citizens entirely bankrupt. The orchestrator, a prominent local investment broker, has been arrested on multiple charges of embezzlement and wire fraud. Victims report experiencing devastating emotional trauma and severe financial distress, with little hope of recovering their stolen funds.",
    "shouldbe_positive_1.txt": "A local man is being celebrated for his extraordinary bravery after diving into the icy waters of the Green River to save a drowning three-year-old child. The toddler, who had wandered away from a nearby park, is expected to make a full recovery thanks to the rapid, selfless, and life-saving intervention of the courageous passerby.",
    "shouldbe_positive_2.txt": "Medical history was made yesterday when a team of specialized surgeons successfully corrected a severe, previously inoperable heart defect in a 16-year-old girl. The groundbreaking 12-hour procedure offers a completely new lease on life for the patient and provides immense hope for thousands of others facing similar rare diagnoses.",
    "shouldbe_positive_3.txt": "Following a devastating house fire that left a local family of five with nothing, neighbors and compassionate strangers rallied together to raise over $100,000 in just one day. Local businesses have also stepped up, donating clothing, furniture, and temporary housing, demonstrating an inspiring and extraordinary level of community spirit.",
    "shouldbe_positive_4.txt": "Cheers erupted at the volunteer search command center when rescue teams finally located the 28-year-old hiker who had been missing in the national park for nearly a week. Despite surviving on minimal rations and battling harsh overnight weather conditions, the hiker was found in remarkably good spirits and has been happily reunited with his tearful family."
}

model = joblib.load(r"../Models/model3.joblib")

for filename, content in texts.items():
    print(filename + " model said: " + model.predict([content])[0])

shouldbe_negative_1.txtmodel said: Positive
shouldbe_negative_2.txtmodel said: Negative
shouldbe_negative_3.txtmodel said: Positive
shouldbe_negative_4.txtmodel said: Negative
shouldbe_positive_1.txtmodel said: Positive
shouldbe_positive_2.txtmodel said: Positive
shouldbe_positive_3.txtmodel said: Positive
shouldbe_positive_4.txtmodel said: Positive


In [5]:
# ocenial gemini
texts = {
    "positive_1": "https://jamesclear.com/saying-no",
    "positive_2": "https://jamesclear.com/zanshin",
    "negative_1": "https://theprint.in/world/thai-woman-charged-for-murder-of-us-diplomat-at-popular-hotel-in-myanmar/2957563/",
    "negative_2": "https://www.theguardian.com/football/2026/jun/11/omar-artan-scandal-gianni-infantino-donald-trump-world-cup",
    "negative_3": "https://www.bbc.com/news/articles/ce8jde20v83o",
    "neutral_1": "https://www.aljazeera.com/sports/liveblog/2026/6/11/live-mexico-vs-south-africa-world-cup-2026",
    "neutral_2": "https://www.bbc.com/sport/football/live/c0myn4dwvzkt",
    "neutral_3": "https://www.bbc.com/news/articles/c0qy0dqldqjo",
    "neutral_4": "https://www.bbc.com/sport/football/articles/c05yz1yr08mo",
    "neutral_5": "https://www.bbc.com/news/articles/c2ly928xz8go"
}
import text_analyser
sa = text_analyser.SentimentAnalyzer(r"../Models/model3.joblib")
for should_be, link in texts.items():
    print(should_be + " model said: " + sa.analyse_article_url(link))

positive_1 model said: Negative
positive_2 model said: Positive
negative_1 model said: Neutral
negative_2 model said: Positive
negative_3 model said: Positive
neutral_1 model said: Positive
neutral_2 model said: Negative
neutral_3 model said: Positive
neutral_4 model said: Positive
neutral_5 model said: Positive
